In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
import torch

# if torch.cuda.is_available():
#     # set default device to GPU
#     torch.set_default_device("cuda")

In [30]:
import torch
from scripts.utils.load_model import load_model, ModelName
from scripts.utils.load_dataset import load_dataset

model, trans = load_model(ModelName.LIU_SWIN, "imagenet", device="cuda:0")
dl_train, dl_eval, dl_test = load_dataset("imagenet", trans, train_batch=128, eval_batch=128)

INFO 03-29 02:07:27 [scripts.utils.load_model_robustbench:104] Model        : Liu2023Comprehensive_Swin_B
INFO 03-29 02:07:27 [scripts.utils.load_model_robustbench:105] Device       : cuda:0
INFO 03-29 02:07:27 [scripts.utils.load_model_robustbench:106] RB Name      : A Comprehensive Study on Robustness of Image Classification Models: Benchmarking and Rethinking
INFO 03-29 02:07:27 [scripts.utils.load_model_robustbench:107] RB Arch      : Swin-B
INFO 03-29 02:07:27 [scripts.utils.load_model_robustbench:108] RB Clean Acc : 76.16 


Creating Dataset:   0%|          | 0/10000 [00:00<?, ?it/s]

Creating Dataset:   0%|          | 0/20000 [00:00<?, ?it/s]

Extracting Samples:   0%|          | 0/10 [00:00<?, ?it/s]

Extracting Samples:   0%|          | 0/10 [00:00<?, ?it/s]

Extracting Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [31]:
from ulib.pert_module import PertModule
from ulib.evaluator import ExtendedEvaluator
from torchattacks.attack import Attack

class AttackedModule(PertModule):
    def __init__(self, iap_attack: Attack, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.iap_attack = iap_attack

    def forward_iap(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """
        Apply the perturbation to the input and pass it through the model.

        Args:
            x (torch.Tensor): Input tensor.
            y (torch.Tensor): Target labels for the input tensor.

        Returns:
            torch.Tensor: Output tensor from the model.
        """
        
        with torch.inference_mode(False):
            x = x.clone().detach().to(self.device)
            y = y.clone().detach().to(self.device)
            x_adv = self.iap_attack(x, labels=y)
            
        return self.model(x_adv.to(self.device))


class IAPEvaluator(ExtendedEvaluator):
    def __init__(self, model: AttackedModule, *args, **kwargs):
        super().__init__(model, *args, **kwargs)

    def eval_batch(self, data: tuple[torch.Tensor, ...]) -> dict[str, torch.Tensor]:
        x, y = data
        pert_module: AttackedModule = self.model  # type: ignore
        pred_cln = pert_module.model(x).argmax(dim=1)
        pred_adv = pert_module.forward_iap(x, y).argmax(dim=1)

        fooled = pred_cln != pred_adv
        clean_correct = pred_cln == y
        robust_correct = pred_adv == y
        clean_incorrect = ~clean_correct
        robust_incorrect = ~robust_correct

        return {
            "FR": fooled.to(torch.float32),
            "C-ACC": clean_correct.to(torch.float32),
            "R-ACC": robust_correct.to(torch.float32),
            "C-MR": clean_incorrect.to(torch.float32),
            "R-MR": robust_incorrect.to(torch.float32),
        }

In [32]:
import torchattacks
from torchattacks.attack import Attack


def attack_builder(model: torch.nn.Module, attack: str, eps: float, steps: int) -> Attack:

    if attack == "fgsm":
        return torchattacks.FGSM(model, eps=eps)
    if attack == "ffgsm":
        return torchattacks.FFGSM(model, eps=eps, alpha=eps)
    if attack == "pgd":
        return torchattacks.PGD(model, eps=eps, alpha=eps / 3, steps=steps)
    if attack == "cw":
        return torchattacks.CW(model, lr=eps / 3, steps=steps)
    if attack == "tpgd":
        return torchattacks.TPGD(model, eps=eps, alpha=eps / 3, steps=steps)
    if attack == "apgd":
        return torchattacks.APGD(model, eps=eps, steps=steps, loss="dlr")
    if attack == "jitter":
        return torchattacks.Jitter(model, eps=eps, alpha=eps / 3, steps=steps)

    raise ValueError(f"Unsupported inner attack method: {attack}")


In [33]:
eps = 16 / 255
steps = 5
attacks = [
    "fgsm",
    "ffgsm",
    "pgd",
    "cw",
    "jitter",
    "apgd",
    "tpgd",
]

for attack in attacks:
    iap = attack_builder(
        model,
        attack=attack,
        eps=eps,
        steps=steps,
    )

    attacked_model = AttackedModule(
        iap_attack=iap,
        model=model,
        data_shape=dl_train.get_tensor(0).shape[1:],
        eps=eps,
    )

    evaluator = IAPEvaluator(
        attacked_model,
        verbose=True,
    )

    print()
    print(f"Attack {attack}: ", evaluator.evaluate(dl_eval))

Evaluating Extended Metrics:   0%|          | 0/79 [00:00<?, ?it/s]

Attack fgsm:  {'FR': 0.4455, 'ASR': 0.4914381357023303, 'C-ACC': 0.7767, 'R-ACC': 0.395, 'C-MR': 0.2233, 'R-MR': 0.605}



Evaluating Extended Metrics:   0%|          | 0/79 [00:00<?, ?it/s]

Attack ffgsm:  {'FR': 0.3964, 'ASR': 0.43285695892880127, 'C-ACC': 0.7767, 'R-ACC': 0.4405, 'C-MR': 0.2233, 'R-MR': 0.5595}



Evaluating Extended Metrics:   0%|          | 0/79 [00:00<?, ?it/s]

...

Attack jitter:  {'FR': 0.5504, 'ASR': 0.6799279000901249, 'C-ACC': 0.7767, 'R-ACC': 0.2486, 'C-MR': 0.2233, 'R-MR': 0.7514}



Evaluating Extended Metrics:   0%|          | 0/79 [00:00<?, ?it/s]

Attack apgd:  {'FR': 0.6041, 'ASR': 0.7777777777777778, 'C-ACC': 0.7767, 'R-ACC': 0.1726, 'C-MR': 0.2233, 'R-MR': 0.8274}



Evaluating Extended Metrics:   0%|          | 0/79 [00:00<?, ?it/s]

Attack tpgd:  {'FR': 0.483, 'ASR': 0.3965495043131196, 'C-ACC': 0.7767, 'R-ACC': 0.4687, 'C-MR': 0.2233, 'R-MR': 0.5313}
